In [ ]:
%run ./fsr_config.py

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, min as spark_min, max as spark_max, length, size, when, isnull

spark = SparkSession.builder.getOrCreate()

# ── Test table overrides (match the values used in Process 1 & 2) ───────────
METADATA_TABLE = f"{POC_CATALOG}.{POC_SCHEMA}.fsr_metadata_registry_test"
CHUNK_TABLE    = f"{POC_CATALOG}.{POC_SCHEMA}.fsr_chunks_test"

results = []  # (test_name, passed, detail)

def check(name, condition, detail=""):
    passed = bool(condition)
    results.append((name, passed, detail))
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {name}" + (f"  — {detail}" if detail else ""))

print(f"Validating:")
print(f"  Metadata: {METADATA_TABLE}")
print(f"  Chunks:   {CHUNK_TABLE}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROCESS 1 — Metadata Registry Validation
# ═══════════════════════════════════════════════════════════════════════════════
print("\n=== Process 1 — Metadata Registry ===")

meta_df = spark.table(METADATA_TABLE)
meta_count = meta_df.count()

# ── 1.1  Table is not empty ──────────────────────────────────────────────────
check("1.1 Metadata table not empty", meta_count > 0, f"{meta_count} rows")

# ── 1.2  No duplicate document_ids ───────────────────────────────────────────
distinct_ids = meta_df.select("document_id").distinct().count()
check("1.2 No duplicate document_ids", distinct_ids == meta_count,
      f"distinct={distinct_ids}, total={meta_count}")

# ── 1.3  All required fields populated ───────────────────────────────────────
required_cols = ["document_id", "volume_path", "pdf_name", "metadata_status", "chunk_status"]
for c in required_cols:
    null_count = meta_df.filter(col(c).isNull() | (col(c) == "")).count()
    check(f"1.3 No nulls in '{c}'", null_count == 0, f"{null_count} nulls")

# ── 1.4  metadata_status values are valid ───────────────────────────────────
valid_meta_statuses = {"pending", "ok", "failed", "skipped"}
actual_statuses = set(row.metadata_status for row in meta_df.select("metadata_status").distinct().collect())
bad_statuses = actual_statuses - valid_meta_statuses
check("1.4 Valid metadata_status values", len(bad_statuses) == 0,
      f"found: {actual_statuses}" + (f", invalid: {bad_statuses}" if bad_statuses else ""))

# ── 1.5  chunk_status values are valid ──────────────────────────────────────
valid_chunk_statuses = {"pending", "processing", "completed", "failed"}
actual_cs = set(row.chunk_status for row in meta_df.select("chunk_status").distinct().collect())
bad_cs = actual_cs - valid_chunk_statuses
check("1.5 Valid chunk_status values", len(bad_cs) == 0,
      f"found: {actual_cs}" + (f", invalid: {bad_cs}" if bad_cs else ""))

# ── 1.6  All 'ok' docs have page_count > 0 ─────────────────────────────────
ok_df = meta_df.filter(col("metadata_status") == "ok")
ok_count = ok_df.count()
ok_no_pages = ok_df.filter((col("page_count").isNull()) | (col("page_count") <= 0)).count()
check("1.6 All 'ok' docs have page_count > 0", ok_no_pages == 0,
      f"{ok_count} ok docs, {ok_no_pages} missing page_count")

# ── 1.7  All 'ok' docs have a title ─────────────────────────────────────────
ok_no_title = ok_df.filter(col("title").isNull() | (col("title") == "")).count()
check("1.7 All 'ok' docs have title", ok_no_title == 0,
      f"{ok_no_title} missing title")

# ── 1.8  Date format check (YYYY-MM-DD) ────────────────────────────────────
date_cols = ["report_issued_date", "outage_start_date", "outage_end_date"]
for dc in date_cols:
    bad_dates = ok_df.filter(
        col(dc).isNotNull() & (col(dc) != "") &
        ~col(dc).rlike(r"^\d{4}-\d{2}-\d{2}$")
    ).count()
    check(f"1.8 Date format '{dc}'", bad_dates == 0,
          f"{bad_dates} non-YYYY-MM-DD values")

# ── 1.9  document_id is deterministic (16 hex chars) ────────────────────────
bad_ids = meta_df.filter(~col("document_id").rlike(r"^[0-9a-f]{16}$")).count()
check("1.9 document_id format (16 hex chars)", bad_ids == 0,
      f"{bad_ids} malformed IDs")

# ── 1.10  No 'failed' docs without error message ───────────────────────────
failed_no_err = meta_df.filter(
    (col("metadata_status") == "failed") &
    (col("metadata_error").isNull() | (col("metadata_error") == ""))
).count()
check("1.10 Failed docs have error messages", failed_no_err == 0,
      f"{failed_no_err} failed without error")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROCESS 2 — Chunk Table Validation
# ═══════════════════════════════════════════════════════════════════════════════
print("\n=== Process 2 — Chunk Table ===")

chunk_df = spark.table(CHUNK_TABLE)
chunk_count = chunk_df.count()

# ── 2.1  Chunk table is not empty ───────────────────────────────────────────
check("2.1 Chunk table not empty", chunk_count > 0, f"{chunk_count} rows")

# ── 2.2  No duplicate chunk_ids ─────────────────────────────────────────────
distinct_chunks = chunk_df.select("chunk_id").distinct().count()
check("2.2 No duplicate chunk_ids", distinct_chunks == chunk_count,
      f"distinct={distinct_chunks}, total={chunk_count}")

# ── 2.3  chunk_index is sequential per document (0-based, no gaps) ──────────
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("document_id").orderBy("chunk_index")
gap_df = chunk_df.withColumn("expected_idx", row_number().over(w) - 1)
gaps = gap_df.filter(col("chunk_index") != col("expected_idx")).count()
check("2.3 chunk_index sequential (no gaps)", gaps == 0,
      f"{gaps} out-of-sequence chunks")

# ── 2.4  chunk_text is not empty ────────────────────────────────────────────
empty_text = chunk_df.filter(col("chunk_text").isNull() | (col("chunk_text") == "")).count()
check("2.4 No empty chunk_text", empty_text == 0, f"{empty_text} empty")

# ── 2.5  chunk_size matches actual text length ─────────────────────────────
size_mismatch = chunk_df.filter(col("chunk_size") != length(col("chunk_text"))).count()
check("2.5 chunk_size matches text length", size_mismatch == 0,
      f"{size_mismatch} mismatches")

# ── 2.6  Embedding dimension is correct ─────────────────────────────────────
null_embeddings = chunk_df.filter(col("embedding").isNull()).count()
check("2.6 No null embeddings", null_embeddings == 0, f"{null_embeddings} nulls")

bad_dim = chunk_df.filter(
    col("embedding").isNotNull() & (size(col("embedding")) != EMBEDDING_DIMENSION)
).count()
check(f"2.6 Embedding dim = {EMBEDDING_DIMENSION}", bad_dim == 0,
      f"{bad_dim} wrong dimension")

# ── 2.7  start_page <= end_page ─────────────────────────────────────────────
bad_pages = chunk_df.filter(col("start_page") > col("end_page")).count()
check("2.7 start_page <= end_page", bad_pages == 0, f"{bad_pages} violations")

# ── 2.8  Chunk sizes within expected bounds ─────────────────────────────────
max_expected = CHUNKING_CONFIG.chunk_size + 200  # small tolerance
oversized = chunk_df.filter(col("chunk_size") > max_expected).count()
check(f"2.8 Chunk size <= {max_expected}", oversized == 0,
      f"{oversized} oversized chunks")

# ── 2.9  chunk_count field matches actual count per doc ─────────────────────
actual_counts = chunk_df.groupBy("document_id").agg(count("*").alias("actual_count"))
count_check = chunk_df.select("document_id", "chunk_count").distinct().join(
    actual_counts, "document_id"
)
count_mismatch = count_check.filter(col("chunk_count") != col("actual_count")).count()
check("2.9 chunk_count matches actual", count_mismatch == 0,
      f"{count_mismatch} mismatches")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CROSS-PROCESS — Metadata ↔ Chunk Consistency
# ═══════════════════════════════════════════════════════════════════════════════
print("\n=== Cross-Process — Metadata ↔ Chunks ===")

# ── 3.1  Every 'completed' doc has chunks ──────────────────────────────────
completed_docs = set(
    row.document_id for row in
    meta_df.filter(col("chunk_status") == "completed").select("document_id").collect()
)
chunked_docs = set(
    row.document_id for row in
    chunk_df.select("document_id").distinct().collect()
)
completed_no_chunks = completed_docs - chunked_docs
check("3.1 All 'completed' docs have chunks", len(completed_no_chunks) == 0,
      f"{len(completed_no_chunks)} completed docs missing chunks")

# ── 3.2  No orphan chunks (chunk doc_id not in metadata) ───────────────────
meta_doc_ids = set(
    row.document_id for row in meta_df.select("document_id").collect()
)
orphan_chunks = chunked_docs - meta_doc_ids
check("3.2 No orphan chunks", len(orphan_chunks) == 0,
      f"{len(orphan_chunks)} orphan document_ids in chunk table")

# ── 3.3  Materialized metadata matches source ──────────────────────────────
# Check that key fields in chunk rows match the metadata registry
join_df = chunk_df.alias("c").join(
    meta_df.alias("m"), col("c.document_id") == col("m.document_id")
).select(
    col("c.document_id"),
    (col("c.esn") != col("m.esn")).alias("esn_mismatch"),
    (col("c.title") != col("m.title")).alias("title_mismatch"),
    (col("c.equipment_type") != col("m.equipment_type")).alias("equip_mismatch"),
    (col("c.page_count") != col("m.page_count")).alias("page_mismatch"),
)

for field_name in ["esn", "title", "equip", "page"]:
    col_name = f"{field_name}_mismatch"
    mismatches = join_df.filter(col(col_name) == True).select("document_id").distinct().count()
    check(f"3.3 {field_name} matches metadata", mismatches == 0,
          f"{mismatches} docs with mismatch")

# ── 3.4  Status alignment: no 'pending' chunk_status for 'ok' metadata ─────
stale = meta_df.filter(
    (col("metadata_status") == "ok") & (col("chunk_status") == "pending")
).count()
check("3.4 No stale pending chunks for 'ok' metadata", stale == 0,
      f"{stale} docs still pending after both processes ran")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
total = len(results)
passed = sum(1 for _, p, _ in results if p)
failed = total - passed

print(f"  TOTAL: {total}  |  PASSED: {passed}  |  FAILED: {failed}")
print("=" * 60)

if failed > 0:
    print("\nFailed tests:")
    for name, p, detail in results:
        if not p:
            print(f"  [FAIL] {name}  — {detail}")
else:
    print("\nAll tests passed.")

# ── Data profile summary ────────────────────────────────────────────────────
print("\n--- Data Profile ---")
print(f"  Metadata rows:    {meta_count}")
print(f"  Chunk rows:       {chunk_count}")
print(f"  Documents chunked: {len(chunked_docs)}")

stats = chunk_df.agg(
    spark_min("chunk_size").alias("min_size"),
    spark_max("chunk_size").alias("max_size"),
    avg("chunk_size").alias("avg_size"),
).first()
print(f"  Chunk size: min={stats.min_size}  max={stats.max_size}  avg={stats.avg_size:.0f}")

display(spark.sql(f"""
    SELECT m.metadata_status, m.chunk_status, COUNT(*) AS doc_count
    FROM {METADATA_TABLE} m
    GROUP BY m.metadata_status, m.chunk_status
    ORDER BY 1, 2
"""))